# OpenProject REST API Exploration Playground
This notebook demonstrates direct interaction with the OpenProject REST v3 API using `httpx` and the configured OpenProject credentials.

In [6]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
import httpx
from src.ragforge.config import OPENPROJECT_URL, OPENPROJECT_API_KEY

print(f"OpenProject URL: {OPENPROJECT_URL}")
auth = ("apikey", OPENPROJECT_API_KEY)

OpenProject URL: http://localhost:8080


## 1. Get Project List

In [3]:
url = f"{OPENPROJECT_URL}/api/v3/projects"
response = httpx.get(url, auth=auth)
response.raise_for_status()

projects = response.json().get("_embedded", {}).get("elements", [])
print("=== OpenProject Projects ===")
for p in projects:
    print(f"ID: {p['id']} | Name: {p['name']} | Identifier: {p['identifier']}")

=== OpenProject Projects ===
ID: 2 | Name: Scrum project | Identifier: your-scrum-project
ID: 1 | Name: Demo project | Identifier: demo-project


## 2. Get Work Packages (Tasks) for Project ID 1

In [4]:
project_id = 1
url = f"{OPENPROJECT_URL}/api/v3/projects/{project_id}/work_packages"
response = httpx.get(url, auth=auth)
response.raise_for_status()

tasks = response.json().get("_embedded", {}).get("elements", [])
print(f"=== OpenProject Tasks for Project {project_id} ===")
for t in tasks:
    status = t.get("_links", {}).get("status", {}).get("title", "N/A")
    print(f"ID: {t['id']} | Subject: {t['subject']} | Status: {status}")

=== OpenProject Tasks for Project 1 ===
ID: 2 | Subject: Organize open source conference | Status: In progress
ID: 3 | Subject: Set date and location of conference | Status: In progress
ID: 4 | Subject: Send invitation to speakers | Status: In progress
ID: 5 | Subject: Contact sponsoring partners | Status: New
ID: 6 | Subject: Create sponsorship brochure and hand-outs | Status: New
ID: 7 | Subject: Invite attendees to conference | Status: New
ID: 8 | Subject: Setup conference website | Status: New
ID: 9 | Subject: Conference | Status: Scheduled
ID: 10 | Subject: Follow-up tasks | Status: To be scheduled
ID: 11 | Subject: Upload presentations to website | Status: New
ID: 12 | Subject: Party for conference supporters :-) | Status: New
ID: 13 | Subject: End of project | Status: New


## 3. Create a Task & Add Comment

In [5]:
# Step A: Create work package (Task)
create_url = f"{OPENPROJECT_URL}/api/v3/projects/{project_id}/work_packages"
payload = {
    "subject": "Jupyter Playground Exploration Task",
    "description": {
        "format": "markdown",
        "raw": "Task created directly from OpenProject Jupyter Notebook explorer."
    },
    "_links": {
        "type": {"href": "/api/v3/types/1"}  # standard Task type
    }
}

res_create = httpx.post(create_url, json=payload, auth=auth)
res_create.raise_for_status()
new_task = res_create.json()
task_id = new_task["id"]
print(f"Created work package successful! New Task ID: {task_id}")

# Step B: Add Comment to Task
comment_url = f"{OPENPROJECT_URL}/api/v3/work_packages/{task_id}/activities"
comment_payload = {
    "comment": {
        "format": "markdown",
        "raw": "Added comment from notebook playground!"
    }
}

res_comment = httpx.post(comment_url, json=comment_payload, auth=auth)
res_comment.raise_for_status()
print("Comment added successfully!")

Created work package successful! New Task ID: 37
Comment added successfully!
